In [1]:
# imports
import numpy as np
import pandas as pd


from collections import defaultdict
from operator import itemgetter
from os import path


In [18]:
# bring in intersections -> DataFrame intersections 
intersection_data_path = "./data/cleaner/intersections_clean.json"

grid_column = "KY_grid_XY"
# I've changed this a few times and it's getting annoying to change it everytime :)

# import intersection data
def read_in_intersections(path_to_intersection_data):
    df = pd.read_json(path_to_intersection_data, orient='records', lines=True)
    df['GEOMETRY'] = df.GEOMETRY.apply(np.array)
    df[grid_column] = df[grid_column].apply(np.array)
    return df.set_index("INTID")       

intersections = read_in_intersections(intersection_data_path).drop([grid_column], axis=1)

#intersections.GEOMETRY = intersections.GEOMETRY.apply(np.array)
#intersections.GEOMETRY.iloc[0] # == array([-85.51043903,  38.20587931]) # good 

intersections.head()

# from common.county_geometry import convert_points

# grid_to_LL = intersections.KY_grid_XY.apply(convert_points.point_to_ll)
# LL_to_grid = intersections.GEOMETRY.apply(convert_points.point_to_grid)



,FST_ROADNAME,SEC_ROADNAME,SIFCODE1,SIFCODE2,FST_SIFID,SEC_SIFID,GEOMETRY
INTID,,,,,,,
5710837346,REHL RD,W REHL CT,5464,7662,4976,6856,"[-85.51044384084946, 38.20588660809318]"
10005800273,REHL RD,TUCKER STATION RD,5464,6551,4976,5908,"[-85.52816882852979, 38.20037561255241]"
14300767569,REHL RD,TUCKER STATION RD,5464,6551,4976,5908,"[-85.52841872335158, 38.200360349057064]"
18011691414,I 64 EAST,I 265 RAMP,3194,9996,3076,8763,"[-85.50495414554669, 38.22260344055154]"
23945910678,I 265 NORTH,I 265 RAMP,9349,9996,8197,8763,"[-85.50554642815675, 38.222127288727606]"


In [23]:
# bring in centerlines -> DataFrame centerlines
centerlines_path = "./data/cleaner/centerlines_clean.json"

def read_in_centerlines(path_to_data):
    df = pd.read_json(path_to_data, orient='records', lines=True)
    return df.set_index("OBJECTID")

centerline_data = read_in_centerlines(centerlines_path)
#centerlines.head()

# fix geometry column

def get_geo_ends(df):
    ends = df.GEOMETRY.transform({"GEOLOW":itemgetter(0), 'GEOHI':itemgetter(-1)})
    df['GEOLOW'] = ends.GEOLOW.apply(np.array)
    df['GEOHI'] = ends.GEOHI.apply(np.array) 
    return df

centerlines = get_geo_ends(centerline_data)

# set aside full geo for now:
centerline_GEOMETRY = centerlines.GEOMETRY
centerlines = centerlines.drop(["GEOMETRY"], axis=1)

display(centerline_GEOMETRY.head())
centerlines.head()


OBJECTID
1    [[-85.6809503218278, 38.158867088749105], [-85...
2    [[-85.80120122370631, 38.23056379303306], [-85...
3    [[-85.80501498815028, 38.228933021550915], [-8...
4    [[-85.68020545343364, 38.24806713661984], [-85...
5    [[-85.74203979766622, 38.211814770925905], [-8...
Name: GEOMETRY, dtype: object

,ROADNAME,SIFID,SIFCODE,low_cross_ROADNAME,SIFIDLOW,LOCROSSSIF,hi_cross_ROADNAME,SIFIDHI,HICROSSSIF,CORE_CLASS,GEOLOW,GEOHI
OBJECTID,,,,,,,,,,,,
1,SERENITY CT,8665,9887,DELLAFAY DR,1550,1557,DEAD END,8594,9811,LOCAL,"[-85.6809503218278, 38.158867088749105]","[-85.68123463490603, 38.158180484404916]"
2,S 28TH ST,5926,6570,W HILL ST,2854,2934,W GAULBERT AVE,2470,2499,LOCAL,"[-85.80120122370631, 38.23056379303306]","[-85.8013692697861, 38.22934335950893]"
3,BEECH ST,473,0458,WILSON AVE,6487,7212,DR WILLIAM G WEATHERS DR,10596,D596,LOCAL,"[-85.80501498815028, 38.228933021550915]","[-85.80483162492965, 38.22753788732054]"
4,GARDEN DR,2442,2470,RAINBOW DR,13394,5391,POPPY WAY,4702,5128,PRIMARY COLLECTOR,"[-85.68020545343364, 38.24806713661984]","[-85.67986300781635, 38.24760815991686]"
5,PARKWAY DR,4573,4974,MOUNT CLAIRE AVE,4162,4476,DEAD END,8594,9811,LOCAL,"[-85.74203979766622, 38.211814770925905]","[-85.74164752771065, 38.21196859682181]"


In [26]:
# remove interstates from centerlines ?\
# - and other roads like ramps?
exclusions = ('EXPRESSWAY', 'INTERSTATE RAMP')
centerlines = centerlines[~centerlines.CORE_CLASS.isin(exclusions)]

# remove these from intersections as well? 

centerlines.head()

,ROADNAME,SIFID,SIFCODE,low_cross_ROADNAME,SIFIDLOW,LOCROSSSIF,hi_cross_ROADNAME,SIFIDHI,HICROSSSIF,CORE_CLASS,GEOLOW,GEOHI
OBJECTID,,,,,,,,,,,,
1,SERENITY CT,8665,9887,DELLAFAY DR,1550,1557,DEAD END,8594,9811,LOCAL,"[-85.6809503218278, 38.158867088749105]","[-85.68123463490603, 38.158180484404916]"
2,S 28TH ST,5926,6570,W HILL ST,2854,2934,W GAULBERT AVE,2470,2499,LOCAL,"[-85.80120122370631, 38.23056379303306]","[-85.8013692697861, 38.22934335950893]"
3,BEECH ST,473,0458,WILSON AVE,6487,7212,DR WILLIAM G WEATHERS DR,10596,D596,LOCAL,"[-85.80501498815028, 38.228933021550915]","[-85.80483162492965, 38.22753788732054]"
4,GARDEN DR,2442,2470,RAINBOW DR,13394,5391,POPPY WAY,4702,5128,PRIMARY COLLECTOR,"[-85.68020545343364, 38.24806713661984]","[-85.67986300781635, 38.24760815991686]"
5,PARKWAY DR,4573,4974,MOUNT CLAIRE AVE,4162,4476,DEAD END,8594,9811,LOCAL,"[-85.74203979766622, 38.211814770925905]","[-85.74164752771065, 38.21196859682181]"


\# Old code to match centerlines and intersections by SIFID pairs

```py
def build_2(intersection_sifid_index, centerline_sifid_index, 
                      intersection_dict, centerline_dict) -> None:
    # build intersection sifid pair / centerline sifid pair index
    matchset = index_by_SIFID_pairs(intersection_sifid_index, centerline_sifid_index)

    for _, intersection_ids, centerline_ids in matchset.itertuples():
        for int_id in intersection_ids:
            # gather centerline ids into appropriate dictionaries, indexed by intersection ids
            for cl_id in centerline_ids:
                intersection_dict[(int_id, cl_id)] = True
                centerline_dict[(int_id, cl_id)] = True
```

This code was an improved version of this:

```py
def build_match_dicts(intersection_sifid_index, centerline_sifid_index, 
                      intersection_dict, centerline_dict) -> None:
    # build intersection sifid pair / centerline sifid pair index
    matchset = index_by_SIFID_pairs(intersection_sifid_index, centerline_sifid_index)

    for _, intersection_ids, centerline_ids in matchset.itertuples():
        for int_id in intersection_ids:
            # gather centerline ids into appropriate dictionaries, indexed by intersection ids
            intersection_dict[int_id].update(centerline_ids)
            centerline_dict[int_id].update(centerline_ids)
```

Both required error-prone finagling of data objects like this:

```py
# create dictionaries to store future series info
intersections_FST_match = defaultdict(set)
intersections_SEC_match = defaultdict(set)
centerlines_LOW_match = defaultdict(set)
centerlines_HI_match = defaultdict(set)

intm1 = dict()#pd.Series(name='FST_match')
intm2 = dict()#pd.Series(name='SEC_match')
clLOWm = dict()#pd.Series(name='SIFIDLOW_match')
clHIm = dict()#pd.Series(name='SIFIDHI_match')

build_2(intersections_by_FST_SEC, centerlines_by_SIFID_SIFIDLOW, intm1, clLOWm)
build_2(intersections_by_FST_SEC, centerlines_by_SIFID_SIFIDHI, intm1, clHIm)
build_2(intersections_by_SEC_FST, centerlines_by_SIFID_SIFIDLOW, intm2, clLOWm)
build_2(intersections_by_SEC_FST, centerlines_by_SIFID_SIFIDHI, intm2, clHIm)
# build step takes 30s! # not anymore very fast now for some reaason.
```


```python
# Create indexes, sort information into the appropriate dictionaries.
build_match_dicts(intersections_by_FST_SEC, centerlines_by_SIFID_SIFIDLOW, intersections_FST_match, centerlines_LOW_match)
    # Where intersections[FST_SIFID] == centerlines[SIFID] & intersections[SEC_SIFID] == centerlines[SIFIDLOW]
    # Map intersection ids to the centerline ids. Next, do the same for all the other combinations of columns:
build_match_dicts(intersections_by_FST_SEC, centerlines_by_SIFID_SIFIDHI, intersections_FST_match, centerlines_HI_match)
build_match_dicts(intersections_by_SEC_FST, centerlines_by_SIFID_SIFIDLOW, intersections_SEC_match, centerlines_LOW_match)
build_match_dicts(intersections_by_SEC_FST, centerlines_by_SIFID_SIFIDHI, intersections_SEC_match, centerlines_HI_match)

# Convert the dictionaries to Series.
intersections_FST_match = pd.Series(intersections_FST_match, name='FST_match')
intersections_SEC_match = pd.Series(intersections_SEC_match, name='SEC_match')
centerlines_LOW_match = pd.Series(centerlines_LOW_match, name='SIFIDLOW_match')
centerlines_HI_match = pd.Series(centerlines_HI_match, name='SIFIDHI_match')

# Create dataframe by concatenating Series. Intersection id's are the index. 
matchdf = pd.concat((intersections_FST_match, intersections_SEC_match, centerlines_LOW_match, centerlines_HI_match), axis=1)
matchdf.info()
```

```py
intm1 = pd.Series(intm1, name='FST_match', dtype='boolean')
intm2 = pd.Series(intm2, name='SEC_match', dtype='boolean')
clLOWm = pd.Series(clHIm, name='SIFIDLOW_match', dtype='boolean')
clHIm = pd.Series(clLOWm, name='SIFIDHI_match', dtype='boolean')

newmatch = pd.concat((intm1, intm2, clLOWm, clHIm), axis=1)
newmatch[newmatch.SIFIDHI_match & newmatch.SIFIDLOW_match]
```

In [27]:
# # look up roadname by sifid

# names = centerlines.groupby("SIFID").ROADNAME.apply(set)
# all(names.apply(len) == 1) # -> True. :)
# sifid_to_roadname = names.apply(set.pop)

# sifid_to_roadname



In [28]:
# get groups
def get_groups(df, by) -> dict:
    return pd.Series(df.groupby(by=by).groups)

centerlines_by_SIFID_SIFIDLOW = get_groups(centerlines, ['SIFID', 'SIFIDLOW'])
centerlines_by_SIFID_SIFIDHI = get_groups(centerlines, ['SIFID', 'SIFIDHI'])

#
#any(centerlines_by_SIFID_SIFIDLOW.apply(len)>1) # True
#any(centerlines_by_SIFID_SIFIDHI.apply(len)>1) # True
#centerlines_by_SIFID_SIFIDHI

intersections_by_FST_SEC = get_groups(intersections, ['FST_SIFID', 'SEC_SIFID'])
# If the index of one of the centerline groups matches an index in this group, that means
#
#   (centerline == centerlines.loc[centerline index])
#   intersection[FST_SIFID] == centerline[SIFID]
#                   and
#   intersections[SEC_SIFID] == either centerline[SIFIDLOW] or centerline[SIFIDHI]
# depending on which set of centerline groups, which we built above, that we are using. 

# The names FST_SIFID / SEC_SIFID -> First / Second intersection SIFID are convention and
# the order is arbitrary. Reversing the order of the index allows us to match SEC_SIFID efficiently.
intersections_by_SEC_FST = get_groups(intersections, ['SEC_SIFID', 'FST_SIFID'])
# If the index of one of the centerline groups is an index in this group, that means
#
#   intersection[SEC_SIFID] == centerline[SIFID]
#                   and
#   intersections[FST_SIFID] == either centerline[SIFIDLOW] or centerline[SIFIDHI]

#
#any(intersections_by_FST_SEC.apply(len) > 1) # True
#any(intersections_by_SEC_FST.apply(len) > 1) # True


def index_by_SIFID_pairs(intersection_sifid_pairs, centerline_sifid_pairs) -> pd.DataFrame:
    return pd.concat((intersection_sifid_pairs, centerline_sifid_pairs), axis=1,
                    # map intersection ids to centerline ids where their sifid pair indexes match
                    # matches are collections of intersection ids/indexes and centerline ids/indexes
                        ).dropna(how='any')
                    # if either collection of ids is empty (or both), there is no match.


In [16]:
mapping = defaultdict(int)

fst_mask = 0b0001
sec_mask = 0b0010
low_mask = 0b0100
hi_mask = 0b1000

for imask, I in ((fst_mask, intersections_by_FST_SEC), (sec_mask, intersections_by_SEC_FST)):
    for cmask, C in ((low_mask, centerlines_by_SIFID_SIFIDLOW), (hi_mask, centerlines_by_SIFID_SIFIDHI)):
        matchset = index_by_SIFID_pairs(I, C)
        set_cols = imask + cmask
        for (s1, s2), intersection_ids, centerline_ids in matchset.itertuples():
            for intersection_id in intersection_ids:
                for cl_id in centerline_ids:
                    mapping[(intersection_id, cl_id)] |= set_cols

mdf = matchdfnumeric = pd.Series(mapping)

matchdfnumeric.index.set_names(('int_id', 'cl_id'), inplace=True)
matchdfnumeric.apply("0b {:04b}".format)

int_id           cl_id 
589991067518338  10250     0b 0101
                 28655     0b 0101
590506463593858  10250     0b 0101
                 28655     0b 0101
409323268213029  26919     0b 0101
                            ...   
847134907955010  174788    0b 1010
847207302035270  175429    0b 1010
847212217628486  175434    0b 1010
847246577432390  175435    0b 1010
847250872399689  175437    0b 1010
Length: 64111, dtype: object

In [18]:
iw = matchdfnumeric.index.to_frame()

intersection_geos = intersections[['GEOMETRY', 'KY_grid_XY']]

def get_int_geo(intid):
    return intersection_geos.loc[intid]

iw.int_id.apply(get_int_geo)

#centerline_geo = centerlines[
centerline_geos = centerline_GEOMETRY.transform({"GEO_start":itemgetter(0), 'GEO_end':itemgetter(-1)})
#centerline_geos

def get_cl_geo(clid):
    return centerline_geos.loc[clid]

In [19]:

int_points = iw.int_id.apply(get_int_geo)
int_points.columns = ['coordinates', 'state_grid_convert']
int_points


coordinates  \
int_id          cl_id                                              
589991067518338 10250    [-85.89203421085367, 38.14392222397775]   
                28655    [-85.89203421085367, 38.14392222397775]   
590506463593858 10250   [-85.89205718950319, 38.143226353738676]   
                28655   [-85.89205718950319, 38.143226353738676]   
409323268213029 26919   [-85.83013514539758, 38.213609743764856]   
...                                                          ...   
847134907955010 174788   [-85.49669357922623, 38.29070651405069]   
847207302035270 175429   [-85.55862850424626, 38.12636761108555]   
847212217628486 175434   [-85.55884906334812, 38.12802620488927]   
847246577432390 175435    [-85.55854188749127, 38.1271646809743]   
847250872399689 175437   [-85.55753279215419, 38.12710321087191]   

                                    state_grid_convert  
int_id          cl_id                                   
589991067518338 10250      [1168205.125, 238698.00125]  
                28655      [1168205.125, 238698.00125]  
590506463593858 10250           [1168194.0, 238444.75]  
                28655           [1168194.0, 238444.75]  
409323268213029 26919            [1186439.5, 263760.5]  
...                                                ...  
847134907955010 174788  [1282611.825, 290363.79000001]  
847207302035270 175429    [1263987.15375, 230770.1225]  
847212217628486 175434   [1263932.29625, 231374.95875]  
847246577432390 175435   [1264016.19125, 231060.00125]  
847250872399689 175437      [1264306.12, 231033.49625]  

[64111 rows x 2 columns]

In [ ]:

cl_points = iw.cl_id.apply(get_cl_geo)
cl_points

In [ ]:
# -> create# NAME_df


# #int_d = intersections[['FST_ROADNAME', 'SEC_ROADNAME']]
# cl_d = centerlines[['ROADNAME', 'SIFIDLOW', 'SIFIDHI']]

# def get_int_names(int_id):
#     int_d.loc[int_id]

# def get_cl_names(cl_id):
#     rw, sl, sh = cl_d.loc[cl_id]
#     sl = sifid_to_roadname.get(sl)
#     sh = sifid_to_roadname.get(sh)
#     return pd.Series({
#         'ROADNAME': rw,
#         'LO_cross': sl,
#         'HI_cross': sh})

# NAME_df = iw.cl_id.apply(lambda id:centerlines.loc[id])
# #matchdfnumeric.index.to_frame()
# #4.9 sec

# NAME_df

In [60]:
cl_dat = centerlines[['SIFID', 'SIFIDLOW', 'SIFIDHI']]

def gc(ci):
    return cl_dat.loc[ci]

sifm = matchdfnumeric.index.to_frame().cl_id.apply(gc)
sifm

any(matchdfnumeric == 0b0011) # none
any(matchdfnumeric == 0b0000) # false # none of these either

data = {"int_match": matchdfnumeric & 0b0011,
        "cl_sifid": sifm.SIFID,
        "low_match": sifm.SIFIDLOW[(matchdfnumeric & low_mask) != 0],
        "hi_match": sifm.SIFIDHI[(matchdfnumeric & hi_mask) != 0]}


#dd = pd.concat((int_ser, sifm.SIFID, low_ser, hi_ser), axis=1, keys=['int_match', 'cl_sifid', 'low_match', 'hi_match']).convert_dtypes()

SIFdf = pd.DataFrame.from_dict(data).convert_dtypes()
SIFdf

int_match  cl_sifid  low_match  hi_match
int_id           cl_id                                           
5710837346       20353           1      4976       6856      <NA>
                 20860           2      6856       4976      <NA>
                 52803           1      4976       <NA>      6856
10005800273      7109            1      4976       5908      5908
                 9233            1      4976       5908      <NA>
...                            ...       ...        ...       ...
2698443752596326 32774           1      5908        351      <NA>
                 105004          1      5908        351      <NA>
2698447711168309 4069            1      4716      12859      <NA>
                 8304            2     12859       4716      <NA>
                 30017           1      4716       <NA>     12859

[64111 rows x 4 columns]

In [61]:
# make multi index coumns helper

# def ix(columns, *labels):
#     out = list()
#     columns = iter(columns)
#     for label, column in zip(labels, columns):
#         out.append((label, column))
#     else:
#         # default = last value of label
#         for column in columns:
#             out.append((label, column))
#     return pd.MultiIndex.from_tuples(out)

# ix(NAME_df.columns, 'hello')

# NAME_df.columns=ix(NAME_df.columns, 'hello')
# NAME_df

In [71]:

from common.county_geometry import *

iw = matchdfnumeric.index.to_frame()

intersections_GEOMETRY = intersections.GEOMETRY
intersections_state_grid_GEO = intersections.state_grid_GEO

centerlines_GEOLOW = centerlines.GEOLOW
centerlines_GEOHI = centerlines.GEOHI



In [ ]:
hav = haversine_distance_ft

intersection_coordinates = iw.int_id.apply(lambda int_id:intersections_GEOMETRY.at[int_id])
intersection_state_grid_conv = iw.int_id.apply(lambda int_id:intersections_state_grid_GEO.at[int_id])

# TODO ...

In [ ]:
ids = mdf.index.to_frame(name=['intersection_ids',"centerline_ids"])

int_long_lat = ids.intersection_ids.apply(lambda intid: intersections_GEO.at[intid])
int_state_grid = ids.intersection_ids.apply(lambda intid: intersections_state_grid.at[intid])

hi_points = ids.centerline_ids.apply(lambda cl_id: centerlines_GEOHI.at[cl_id])
low_points = ids.centerline_ids.apply(lambda cl_id: centerlines_GEOLOW.at[cl_id])

int_long_lat.combine(hi_points, central_angle)
int_state_grid.combine(hi_points, central_angle)

int_long_lat.combine(hi_points, central_angle)
int_state_grid.combine(hi_points, central_angle)

int_id           cl_id 
589991067518338  10250     2.542220e-05
                 28655     4.687739e-05
590506463593858  10250     2.816241e-05
                 28655     4.542790e-05
409323268213029  26919     3.775007e-05
                               ...     
847134907955010  174788    1.436194e-07
847207302035270  175429    1.431915e-07
847212217628486  175434    1.431968e-07
847246577432390  175435    1.431938e-07
847250872399689  175437    1.431925e-07
Length: 64111, dtype: float64

In [ ]:

hi_dist = (int_points - hi_points).apply(euclidean_distance)

low_dist = (int_points - low_points).apply(euclidean_distance)


In [ ]:

def encode_closer_point(difference):
    if difference < 0: # hi_dist < low_dist
        return 'hi'
    elif difference > 0: # hi_dist > low_dist
        return 'low'
    elif difference == 0: # hi_dist == low_dist
        return 'both'
    else:
        return pd.NA

dist_diff = (hi_dist - low_dist)
geo_end = dist_diff.apply(encode_closer_point)


geo_end.hasnans # -> False.
# checking something;
(geo_end[geo_end != 'low'] == geo_end[(geo_end == 'hi') | (geo_end == 'both')]).all() # -> True

In [ ]:
close_points = pd.concat((
    hi_points[geo_end != 'low'], 
    low_points[geo_end == 'low']))

close_distance = pd.concat((
    hi_dist[geo_end != 'low'],
    low_dist[geo_end == 'low']))

far_points = pd.concat((
    hi_points[geo_end == 'low'],
    low_points[geo_end == 'hi']))

far_distance = pd.concat((
    hi_dist[geo_end == 'low'],
    low_dist[geo_end == 'hi']))


working = pd.DataFrame(pd.Series(int_points, name='int_point'))
working['geo_end'] = geo_end
working['close_point'] = close_points
working['far_point'] = far_points
working['close_dist'] = close_distance
working['far_dist'] = far_distance
working['dist_diff'] = dist_diff.abs()

working.index.set_names(('int_id', 'cl_id'), inplace=True)
geo_data = working

geo_data

int_point geo_end  \
int_id          cl_id                                                      
589991067518338 10250    [-85.89203421085367, 38.14392222397775]     low   
                28655    [-85.89203421085367, 38.14392222397775]     low   
590506463593858 10250   [-85.89205718950319, 38.143226353738676]     low   
                28655   [-85.89205718950319, 38.143226353738676]     low   
409323268213029 26919   [-85.83013514539758, 38.213609743764856]     low   
...                                                          ...     ...   
847134907955010 174788   [-85.49669357922623, 38.29070651405069]      hi   
847207302035270 175429   [-85.55862850424626, 38.12636761108555]      hi   
847212217628486 175434   [-85.55884906334812, 38.12802620488927]      hi   
847246577432390 175435    [-85.55854188749127, 38.1271646809743]      hi   
847250872399689 175437   [-85.55753279215419, 38.12710321087191]      hi   

                                                     close_point  \
int_id          cl_id                                              
589991067518338 10250    [-85.89203421085367, 38.14392222397775]   
                28655   [-85.89205718950319, 38.143226353738676]   
590506463593858 10250    [-85.89203421085367, 38.14392222397775]   
                28655   [-85.89205718950319, 38.143226353738676]   
409323268213029 26919   [-85.83013514539758, 38.213609743764856]   
...                                                          ...   
847134907955010 174788   [-85.49669357922623, 38.29070651405069]   
847207302035270 175429   [-85.55862850424626, 38.12636761108555]   
847212217628486 175434   [-85.55884906334812, 38.12802620488927]   
847246577432390 175435    [-85.55854188749127, 38.1271646809743]   
847250872399689 175437   [-85.55753279215419, 38.12710321087191]   

                                                       far_point  close_dist  \
int_id          cl_id                                                          
589991067518338 10250    [-85.89017790736897, 38.14387530855226]    0.000000   
                28655    [-85.89535933539815, 38.14331916493513]    0.000696   
590506463593858 10250    [-85.89017790736897, 38.14387530855226]    0.000696   
                28655    [-85.89535933539815, 38.14331916493513]    0.000000   
409323268213029 26919    [-85.83201078511965, 38.21202291188871]    0.000000   
...                                                          ...         ...   
847134907955010 174788  [-85.49758979847759, 38.290228130801005]    0.000000   
847207302035270 175429    [-85.55854188749127, 38.1271646809743]    0.000000   
847212217628486 175434   [-85.55899003824544, 38.12895721169443]    0.000000   
847246577432390 175435   [-85.55884906334812, 38.12802620488927]    0.000000   
847250872399689 175437  [-85.55664071193948, 38.127332178618765]    0.000000   

                        far_dist  dist_diff  
int_id          cl_id                        
589991067518338 10250   0.001857   0.001857  
                28655   0.003379   0.002683  
590506463593858 10250   0.001988   0.001292  
                28655   0.003303   0.003303  
409323268213029 26919   0.002457   0.002457  
...                          ...        ...  
847134907955010 174788  0.001016   0.001016  
847207302035270 175429  0.000802   0.000802  
847212217628486 175434  0.000942   0.000942  
847246577432390 175435  0.000915   0.000915  
847250872399689 175437  0.000921   0.000921  

[64111 rows x 7 columns]

In [ ]:

def get_intersection_names(intersection_id):
    in1 = intersections.at[intersection_id, 'FST_ROADNAME']
    in2 = intersections.at[intersection_id, 'SEC_ROADNAME']
    return in1, in2

names = centerlines.groupby("SIFID").ROADNAME.apply(set)#.apply(lambda x:len(x)==1).all() # true
sifid_names = names.apply(lambda x:x.pop())

def roadname_by_sifid(sifid):
    return sifid_names[sifid]

def get_names(int_id, cd_id):
    inames = get_intersection_names(intersection_id)
    cnames = roadname_by_sifid(cl_id)
    return {'intersection':inames, 'centerline':cnames}



In [ ]:
intids = working.index.to_frame().int_id
intids

int_id           cl_id 
589991067518338  10250     589991067518338
                 28655     589991067518338
590506463593858  10250     590506463593858
                 28655     590506463593858
409323268213029  26919     409323268213029
                                ...       
847134907955010  174788    847134907955010
847207302035270  175429    847207302035270
847212217628486  175434    847212217628486
847246577432390  175435    847246577432390
847250872399689  175437    847250872399689
Name: int_id, Length: 64111, dtype: int64

In [ ]:
working = geo_data[geo_data.close_dist >= .1]
working

iw = working.index.to_frame()
iw

def get_int_data(int_id):
    return intersections.loc[int_id][["FST_ROADNAME", "FST_SIFID", "SEC_ROADNAME", "SEC_SIFID"]]

def get_cl_data(cl_id):
    return centerlines.loc[cl_id][['ROADNAME', 'SIFID', 'CORE_CLASS']]

data = pd.concat((
iw.cl_id.transform({"centerline_data":get_cl_data}) ,
iw.int_id.transform({'intersection_data':get_int_data})
), axis=1)

data


def mmft(label, df):
    if isinstance(label, str):
        return pd.MultiIndex.from_tuples((label, col) for col in df.columns)
    else:
        return pd.MultiIndex.from_tuples((L, C) for L, C in zip(label, df.columns))
    


def dd(geodf):
    #ex = geodf[['int_point
    #out = list()
    iw = geodf.index.to_frame()
    cl_geo = geodf[['int_point', 'close_point', 'close_dist', 'geo_end']]
    cl_geo.columns = mmft(('int_geo', 'cl_geo', 'cl_geo', 'cl_geo'), cl_geo)

    cl_data = iw.cl_id.apply(get_cl_data)
    cl_data.columns = mmft('cl_data', cl_data)
    # TODO get hi/low sifid / roadname

    int_data = iw.int_id.apply(get_int_data)
    int_data.columns = mmft("int_data", int_data)

    out = [int_data, cl_geo, cl_data]
    return pd.concat(out, axis=1)
    
    int_data = iw.int_id.apply(get_int_data)
    int_info = pd.concat((int_data, geodf.int_point), axis=1)
    int_info.columns = pd.MultiIndex.from_tuples(('int_data', col) for col in int_info.columns)
    
    return pd.concat((int_info, cl_info), axis=1).sort_index()

#    pd.MultiIndex.from_tuples([('cl_data', col) for col in c.columns])
#c.columns = pd.MultiIndex.from_tuples([('cl_data', col) for col in c.columns])
#c

see = dd(working)

#see[~see.int_data.FST_ROADNAME.str.contains('841')]

see

int_data                                    \
                          FST_ROADNAME FST_SIFID  SEC_ROADNAME SEC_SIFID   
int_id           cl_id                                                     
590467808889140  29463  NO STREET NAME         1   CANE RUN RD       906   
38521867941168   6218     COLUMBIA AVE      1245  KENTUCKY AVE      3356   
589634891494704  21710    COLUMBIA AVE      1245  KENTUCKY AVE      3356   
706470196341142  903       KY-841 RAMP     14021        KY 841     14195   
                 18413     KY-841 RAMP     14021        KY 841     14195   
...                                ...       ...           ...       ...   
654251992213784  6590           KY 841     14195   KY-841 RAMP     14021   
                 7706           KY 841     14195   KY-841 RAMP     14021   
2696491761592600 6590           KY 841     14195   KY-841 RAMP     14021   
                 7706           KY 841     14195   KY-841 RAMP     14021   
                 11399          KY 841     14195   KY-841 RAMP     14021   

                                                         int_geo  \
                                                       int_point   
int_id           cl_id                                             
590467808889140  29463   [-85.89679114783614, 38.14336979587274]   
38521867941168   6218    [-85.61845036330493, 38.26000799417955]   
589634891494704  21710  [-85.84807001201702, 38.150808026597076]   
706470196341142  903    [-85.70246758147974, 38.115233764331414]   
                 18413  [-85.70246758147974, 38.115233764331414]   
...                                                          ...   
654251992213784  6590   [-85.87003129510553, 38.091380651748494]   
                 7706   [-85.87003129510553, 38.091380651748494]   
2696491761592600 6590    [-85.87691759786703, 38.09294723096209]   
                 7706    [-85.87691759786703, 38.09294723096209]   
                 11399   [-85.87691759786703, 38.09294723096209]   

                                                          cl_geo             \
                                                     close_point close_dist   
int_id           cl_id                                                        
590467808889140  29463   [-85.8175947343509, 38.218723744753156]   0.109317   
38521867941168   6218   [-85.84807001201702, 38.150808026597076]   0.254263   
589634891494704  21710   [-85.61845036330493, 38.26000799417955]   0.254263   
706470196341142  903     [-85.8189716474408, 38.103260224945146]   0.117118   
                 18413  [-85.87003129510553, 38.091380651748494]   0.169253   
...                                                          ...        ...   
654251992213784  6590   [-85.75479831645234, 38.117411563113556]   0.118137   
                 7706    [-85.74797696339039, 38.11669772522921]   0.124652   
2696491761592600 6590   [-85.75479831645234, 38.117411563113556]   0.124546   
                 7706    [-85.74797696339039, 38.11669772522921]   0.131110   
                 11399   [-85.77549954907987, 38.11946730096766]   0.104828   

                                     cl_data                         
                       geo_end      ROADNAME  SIFID      CORE_CLASS  
int_id           cl_id                                               
590467808889140  29463      hi       NO NAME      1           LOCAL  
38521867941168   6218      low  COLUMBIA AVE   1245           LOCAL  
589634891494704  21710     low  COLUMBIA AVE   1245           LOCAL  
706470196341142  903       low   KY-841 RAMP  14021  MAJOR ARTERIAL  
                 18413     low   KY-841 RAMP  14021  MAJOR ARTERIAL  
...                        ...           ...    ...             ...  
654251992213784  6590       hi   KY-841 RAMP  14021  MAJOR ARTERIAL  
                 7706      low   KY-841 RAMP  14021  MAJOR ARTERIAL  
2696491761592600 6590       hi   KY-841 RAMP  14021  MAJOR ARTERIAL  
                 7706      low   KY-841 RAMP  14021  MAJOR ARTERI

In [ ]:

c = iw.cl_id.apply(get_cl_data)
i = iw.int_id.apply(get_int_data)


df=pd.DataFrame({'a':[1,2,3],'b':[4,5,6]})

columns=[('c','a'),('c','b')]

df.columns=pd.MultiIndex.from_tuples(columns)
df

pd.MultiIndex.from_tuples([('cl_data', col) for col in c.columns])
c.columns = pd.MultiIndex.from_tuples([('cl_data', col) for col in c.columns])
c

cl_data                       
                            ROADNAME  SIFID      CORE_CLASS
int_id           cl_id                                     
590467808889140  29463       NO NAME      1           LOCAL
38521867941168   6218   COLUMBIA AVE   1245           LOCAL
589634891494704  21710  COLUMBIA AVE   1245           LOCAL
706470196341142  903     KY-841 RAMP  14021  MAJOR ARTERIAL
                 18413   KY-841 RAMP  14021  MAJOR ARTERIAL
...                              ...    ...             ...
654251992213784  6590    KY-841 RAMP  14021  MAJOR ARTERIAL
                 7706    KY-841 RAMP  14021  MAJOR ARTERIAL
2696491761592600 6590    KY-841 RAMP  14021  MAJOR ARTERIAL
                 7706    KY-841 RAMP  14021  MAJOR ARTERIAL
                 11399   KY-841 RAMP  14021  MAJOR ARTERIAL

[89 rows x 3 columns]

In [ ]:
s= working.index.get_level_values('cl_id')

centerlines.loc[s].ROADNAME.value_counts()

ROADNAME
KY-841 RAMP     39
KY 841          36
COLUMBIA AVE     4
CANE RUN RD      4
KENTUCKY AVE     4
NO NAME          2
Name: count, dtype: int64

In [ ]:
mi = intersections.loc[working.index.get_level_values('int_id')]

# KY 841 / I 265 / Watterson Expwy and ramps
mi[(mi.FST_ROADNAME.str.contains("841") & mi.SEC_ROADNAME.str.contains("841"))]

# other weird matches
wm = mi[~(mi.FST_ROADNAME.str.contains("841") & mi.SEC_ROADNAME.str.contains("841"))]

working.loc[wm.index, :]
wm
mi


,FST_ROADNAME,SEC_ROADNAME,SIFCODE1,SIFCODE2,FST_SIFID,SEC_SIFID,GEOMETRY,state_grid_GEO
int_id,,,,,,,,
590467808889140,NO STREET NAME,CANE RUN RD,0000,0934,1,906,"[-85.89679114783614, 38.14336979587274]","[-85.89673981756692, 38.143358031236794]"
38521867941168,COLUMBIA AVE,KENTUCKY AVE,1241,3530,1245,3356,"[-85.61845036330493, 38.26000799417955]","[-85.61844551689906, 38.260000695318304]"
589634891494704,COLUMBIA AVE,KENTUCKY AVE,1241,3530,1245,3356,"[-85.84807001201702, 38.150808026597076]","[-85.84805632815548, 38.15080051575182]"
706470196341142,KY-841 RAMP,KY 841,E918,E996,14021,14195,"[-85.70246758147974, 38.115233764331414]","[-85.70246272131506, 38.11522649717792]"
706470196341142,KY-841 RAMP,KY 841,E918,E996,14021,14195,"[-85.70246758147974, 38.115233764331414]","[-85.70246272131506, 38.11522649717792]"
...,...,...,...,...,...,...,...,...
654251992213784,KY 841,KY-841 RAMP,E996,E918,14195,14021,"[-85.87003129510553, 38.091380651748494]","[-85.87002638784153, 38.09137339695274]"
654251992213784,KY 841,KY-841 RAMP,E996,E918,14195,14021,"[-85.87003129510553, 38.091380651748494]","[-85.87002638784153, 38.09137339695274]"
2696491761592600,KY 841,KY-841 RAMP,E996,E918,14195,14021,"[-85.87691759786703, 38.09294723096209]","[-85.87691268847966, 38.09293997618528]"


In [ ]:


    

#display(
#info[['int_point', 'close_point', 'close_dist', 'geo_end', ]])



wmm = data[~data.intersection_data.FST_ROADNAME.str.contains('841')]

display(
working.loc[wmm.index][['int_point', 'close_point', 'close_dist', 'geo_end', ]],
wmm)


,,int_point,close_point,close_dist,geo_end
int_id,cl_id,,,,
590467808889140,29463,"[-85.89679114783614, 38.14336979587274]","[-85.8175947343509, 38.218723744753156]",0.109317,hi
38521867941168,6218,"[-85.61845036330493, 38.26000799417955]","[-85.84807001201702, 38.150808026597076]",0.254263,low
589634891494704,21710,"[-85.84807001201702, 38.150808026597076]","[-85.61845036330493, 38.26000799417955]",0.254263,low
352187318274356,31519,"[-85.81503148910512, 38.21843165728323]","[-85.89535933539815, 38.14331916493513]",0.109975,low
38521867941168,10508,"[-85.61845036330493, 38.26000799417955]","[-85.84620751970706, 38.15069748385449]",0.252630,low
589634891494704,11695,"[-85.84807001201702, 38.150808026597076]","[-85.61953362264914, 38.25940896160892]",0.253028,low
352187318274356,11220,"[-85.81503148910512, 38.21843165728323]","[-85.89679114783614, 38.14336979587274]",0.110991,low
590467808889140,26722,"[-85.89679114783614, 38.14336979587274]","[-85.81797210922643, 38.21544151873681]",0.106803,hi
38521867941168,2474,"[-85.61845036330493, 38.26000799417955]","[-85.84807001201702, 38.150808026597076]",0.254263,low


centerline_data                           \
                             ROADNAME SIFID         CORE_CLASS   
int_id          cl_id                                            
590467808889140 29463         NO NAME     1              LOCAL   
38521867941168  6218     COLUMBIA AVE  1245              LOCAL   
589634891494704 21710    COLUMBIA AVE  1245              LOCAL   
352187318274356 31519         NO NAME     1              LOCAL   
38521867941168  10508    COLUMBIA AVE  1245              LOCAL   
589634891494704 11695    COLUMBIA AVE  1245              LOCAL   
352187318274356 11220     CANE RUN RD   906  PRIMARY COLLECTOR   
590467808889140 26722     CANE RUN RD   906     MINOR ARTERIAL   
38521867941168  2474     KENTUCKY AVE  3356              LOCAL   
589634891494704 29055    KENTUCKY AVE  3356              LOCAL   
352187318274356 32589     CANE RUN RD   906  PRIMARY COLLECTOR   
590467808889140 5361      CANE RUN RD   906     MINOR ARTERIAL   
38521867941168  10465    KENTUCKY AVE  3356              LOCAL   
589634891494704 5523     KENTUCKY AVE  3356              LOCAL   

                      intersection_data                                    
                           FST_ROADNAME FST_SIFID  SEC_ROADNAME SEC_SIFID  
int_id          cl_id                                                      
590467808889140 29463    NO STREET NAME         1   CANE RUN RD       906  
38521867941168  6218       COLUMBIA AVE      1245  KENTUCKY AVE      3356  
589634891494704 21710      COLUMBIA AVE      1245  KENTUCKY AVE      3356  
352187318274356 31519    NO STREET NAME         1   CANE RUN RD       906  
38521867941168  10508      COLUMBIA AVE      1245  KENTUCKY AVE      3356  
589634891494704 11695      COLUMBIA AVE      1245  KENTUCKY AVE      3356  
352187318274356 11220    NO STREET NAME         1   CANE RUN RD       906  
590467808889140 26722    NO STREET NAME         1   CANE RUN RD       906  
38521867941168  2474       COLUMBIA AVE      1245  KENTUCKY AVE      3356  
589634891494704 29055      COLUMBIA AVE      1245  KENTUCKY AVE      3356  
352187318274356 32589    NO STREET NAME         1   CANE RUN RD       906  
590467808889140 5361     NO STREET NAME         1   CANE RUN RD       906  
38521867941168  10465      COLUMBIA AVE      1245  KENTUCKY AVE      3356  
589634891494704 5523       COLUMBIA AVE      1245  KENTUCKY AVE      3356

In [ ]:
centerlines[centerlines.ROADNAME.str.contains("KENTUCKY AVE")]
centerlines[(centerlines.SIFID == 3356) & ((centerlines.SIFIDLOW == 1245) | (centerlines.SIFIDHI == 1245))]

#wm = wm.groupby(by="FST_SIFID").apply(lambda x:x)


,ROADNAME,SIFID,SIFCODE,low_cross_ROADNAME,SIFIDLOW,LOCROSSSIF,hi_cross_ROADNAME,SIFIDHI,HICROSSSIF,CORE_CLASS,GEOLOW,GEOHI
OBJECTID,,,,,,,,,,,,
2474,KENTUCKY AVE,3356,3530,COLUMBIA AVE,1245,1241,PRINCETON AVE,4725,5155,LOCAL,"[-85.84807001201702, 38.150808026597076]","[-85.84822119095742, 38.149025980797525]"
5523,KENTUCKY AVE,3356,3530,FLORIDA AVE,2249,2251,COLUMBIA AVE,1245,1241,LOCAL,"[-85.6148949649379, 38.25574321445772]","[-85.61845036330493, 38.26000799417955]"
10465,KENTUCKY AVE,3356,3530,DEAD END,8594,9811,COLUMBIA AVE,1245,1241,LOCAL,"[-85.84789878936502, 38.1526162694713]","[-85.84807001201702, 38.150808026597076]"
29055,KENTUCKY AVE,3356,3530,COLUMBIA AVE,1245,1241,DEAD END,8594,9811,LOCAL,"[-85.61845036330493, 38.26000799417955]","[-85.62012324704497, 38.25992290121035]"


In [ ]:
def convert_geo(geo):
    if len(geo) == 2:
        long, lat = geo
        return (lat, long)
    else:
        return [(lat, long) for long, lat in geo]

def swap_point(point):
    x, y = point
    return (y, x)

swap_point((1,2))
        

(2, 1)

In [ ]:
# working = mdf.index.to_series()


# def get_geo(row):
#     int_id, cl_id = row
#     out = dict() # tried Series. Took too long. Dict is very fast ( .4 sec)
#     out['intx_geo'] = intx_geo = intersections_GEO.at[int_id]
#     out['cl_geo_low'] = centerlines_GEOLOW.at[cl_id]
#     out['cl_geo_hi'] = centerlines_GEOHI.at[cl_id]
#     return out

# working = pd.DataFrame(mdf.index.to_series().apply(get_geo).to_list(), index=mdf.index)
# hi_dist = (working.intx_geo - working.cl_geo_hi).apply(euclidean_distance)
# low_dist = (working.intx_geo - working.cl_geo_low).apply(euclidean_distance)

# def find_close_closer_point(x):
#     if x < 0:
#         return 'hi'
#     elif x > 0:
#         return 'low'
#     elif x == 0:
#         return 'both'
#     else:
#         return pd.NA

# working['geo_end'] = (hi_dist - low_dist).apply(find_close_closer_point)
# working

# def gp(row):
#     code = row.geo_end
#     if code == 'low':
#         return (row.cl_geo_low, row.cl_geo_hi)
#     elif code == 'hi':
#         return (row.cl_geo_hi,row.cl_geo_low)
#     elif code == 'both':
#         return (row.cl_geo_hi, None)
#     else:
#         return (None, None)

# ee = pd.DataFrame(working.apply(gp, axis=1).to_list(), index=working.index, columns=['close_point', 'far_point'])
# pd.concat((working, ee), axis=1)


In [ ]:
# old versions of code
#     lowdist = euclidean_distance(intx_geo - cl_geo_low)
#     hidist = euclidean_distance(intx_geo - cl_geo_hi)
#     if lowdist < hidist:
#         geo_close = 'low'
#         closer_point = cl_geo_low
#         farther_point = cl_geo_hi
#         close_dist = lowdist
#         far_dist = hidist
#     elif lowdist > hidist:
#         geo_close = 'hi'
#         closer_point = cl_geo_hi
#         farther_point = cl_geo_low
#         close_dist = lowdist
#         far_dist = hidist
#     elif lowdist == hidist:
#         geo_close = 'both'
#         #assert cl_geo_low == cl_geo_hi
#         closer_point = cl_geo_low
#         farther_point = pd.NA
#         close_dist = lowdist
#         far_dist = hidist

#     out['geo_close'] = geo_close
#     out['close_point'] = closer_point
#     out['far_point'] = farther_point
#     out['close_dist'] = close_dist
#     out['far_dist'] = far_dist

#     return out


# working = pd.DataFrame(mdf.index.to_series().apply(find_close_closer_point).to_list(), index=mdf.index)
# #centerlines.loc[working[working.closer_code == 'both'].index.get_level_values(1)] # closer_code = 'both'


In [ ]:

diffs = (working['close_dist'] - working['far_dist']).abs()
# some values are zero
# drop these to make finding min easier
dd = diffs.drop(diffs[diffs == 0].index)


dd.min()

np.float64(0.0005881797990768545)

In [ ]:
# Problematic numbers?

#intersections.loc[319438198302071]
#centerlines.loc[22602]

#centerlines.loc[[84838, 31468]]
#(-85.5284187234, 38.2003603491)
#intersections.loc[[14300767569, 10005800273]]
# Problematic numbers?


#centerlines.loc[[13164, 24805, 26558]],
#intersections.loc[389674641274662])

#low_match_1[low_match_1 == 389674641274662]

# cl, ix =88673, {334354632500584, 726302145904921}


# 5 have 3 matches

# removing ramps from centerlines removed 2 of these

# 11577    {318119662229912, 318128252164504, 31811536726...
# Arthur St. and I 65 RAMP

# 79773    {731803473713560, 844908543362584, 84487847859...
# I 65 ramp

# 80073    {844438991051160, 731813464524312, 73181635861...
# I 65 ramp

# 31660    {582161378284545, 582191443055617, 61834218278...

#582161378284545	AUTUMN WAY	244	SUMMERTIME PKY	8232	(-85.8735529704, 38.0723544119)
#582191443055617	AUTUMN WAY	244	SUMMERTIME PKY	8232	(-85.8741780941, 38.0721878176)
#618342182786049	AUTUMN WAY	244	SUMMERTIME PKY	8232	(-85.873007987, 38.0733107806)

# 19920    {731744745003415, 722828392896919, 72287563753...
# I 65 RAMP



# 1 has 4 matches -> 
# removing EXPRESSWAYS from centerlines dealt with this one.

# 79772 : {731803473713560, 731813464524312, 844878478591512, 844908543362584}
# I 65 NORTH x I 65 RAMP
# no more from there 


# potential problematic numbers

#set(hi_match.keys()).intersection(hi_match2.keys()) # empty : good news
#  14288: ('start', 360168983520312, 8.767631726955189e-06),

#display(
##centerlines.loc[14288],
#intersections.loc[[360168983520312, 360748804105272]]
#

#centerlines.loc[[52803, 20860]]
#centerlines[centerlines.SIFIDLOW == 8594]
#centerlines.loc[[ 1498,  2541,  2610,  3823,  3882,  5039,  6722,  7769,  8692, 11961,
#       14175, 15423, 19704, 23933, 24872, 27737, 29258, 30870]] # I 265 RAMP.
#intersections.loc[847265632743817]
#intersections.loc[722828392896919]
#centerlines.loc[19920]
#ICpairs.loc[6146]

In [ ]:
# roadways = centerlines.groupby('SIFID').groups
# intxn_by_fst_sifid = intersections.groupby('FST_SIFID').groups
# # intxn_by_sec_sifid = intersections.groupby('SEC_SIFID') # probably not necessary
#     # ... since all records will be accessed by iterating over the fst_sifid groupby

# #intersections.FST_SIFID.hasnans # == False this is good

# def map_intersection_ids_to_centerline_ids(intersection_sifid_groups=intxn_by_fst_sifid, centerline_sifid_groups=roadways):
#     found = dict()
#     notfound = list()
#     for sifid_1 in intersection_sifid_groups.keys():
#         roadway_ids = centerline_sifid_groups.get(sifid_1, None)
#         if roadway_ids is not None:
#             found[sifid_1] = roadway_ids # first sifid match to centerlines id
#         else:
#             notfound.append(sifid_1)
#     return found, notfound

# def map_intersections_to_centerlines(intersections=intersections, centerlines=centerlines):
#     mapping = pd.DataFrame(columns=['full_match', 'fst_match_only', 'sec_match_only'])
#     full_matches = pd.Series(name='full_match', dtype="O")
#     roadways = centerlines.groupby('SIFID').groups
#     intersections_by_first_sifid = intersections.groupby('FST_SIFID').groups
#     notfound = list()

#     for sifid_1, intersection_ids in intersections_by_first_sifid.items():
#         centerline_ids = roadways.get(sifid_1, None)
#         if centerline_ids is None:
#             notfound.extend(intersection_ids)
#         else:
#             fst_match = centerlines.loc[centerline_ids]
#             for intersection_id in intersection_ids:
#                 sifid_2 = intersections.at[intersection_id, 'SEC_SIFID']
#                 full_match = fst_match[(fst_match.SIFIDLOW == sifid_2) | (fst_match.SIFIDHI == sifid_2)]
#                 if full_match.empty:
#                     mapping.at[intersection_id, 'fst_match_only'] = True
#                 else:
#                     full_match = full_match.index.tolist()
#                     #print(full_match)
#                     full_matches.at[intersection_id] = full_match
        
#     if notfound:
#         centerline_sifids = roadways.keys()
#         for intersection_id, sifid_2 in intersections.loc[notfound]['SEC_SIFID'].items():
#             if sifid_2 in centerline_sifids:
#                 mapping.at[intersection_id, 'sec_match_only'] = True

#     return pd.concat((full_matches, mapping))


# mapping = map_intersections_to_centerlines()


In [ ]:
#mapping.isna().apply(any, axis=1).all()
# test that each row has at least one value filled -> Yes
#intersections.index.difference(mapping.index) # == Index([693211604576105], dtype='int64')

#display(intersections.loc[693211604576105]) -> 13554, 13555 fst, sec ids

#centerlines[centerlines.SIFID == 13555] # nope, nor 13554 
# probably just ignore this.

#mapping[mapping.full_match.notna()]
#mapping

In [ ]:
# mapping = pd.DataFrame(columns=['full_match', 'fst_match_only', 'sec_match_only'])
# roadways = centerlines.groupby('SIFID').groups
# intersections_by_first_sifid = intersections.groupby('FST_SIFID').groups
# notfound = list()

# for sifid_1, intersection_ids in intersections_by_first_sifid.items():
#     centerline_ids = roadways.get(sifid_1, None)
#     if centerline_ids is None:
#         notfound.extend(intersection_ids)
#     else:
#         fst_match = centerlines.loc[centerline_ids]
#         #print(fst_match)
#         for intersection_id in intersection_ids:
#             sifid_2 = intersections.at[intersection_id, 'SEC_SIFID']
#             full_match = fst_match[(fst_match.SIFIDLOW == sifid_2) | (fst_match.SIFIDHI == sifid_2)]



In [ ]:
centerlines[centerlines.SIFID==1178]
notfound = [624,626,689,903,2053,122932,139877,161371,161372,168074]

intersections[intersections.FST_SIFID == 1178]
nf1 = intersections[intersections.FST_SIFID.isin(notfound)]
nf2 = centerlines[centerlines.SIFID.isin(nf1.SEC_SIFID.values)] # looks to be all interstate ramps
# it makes sense that these would not be in the centerline data b/c I probably stripped them out at a some point
# you can't/shouldn't ride a bicycle on the ramps/interstate!

noramp = nf2[nf2.CORE_CLASS != "INTERSTATE RAMP"] # 205 ramps. We don't need these roadways.
#display(noramp)
#nf.groupby('CORE_CLASS').count()

#nf.groupby('SIFID').groups
#sifid_twos = intersections[intersections.FST_SIFID.isin(notfound)].SEC_SIFID.values#.groupby('SEC_SIFID').groups
#for sifid_2 in sifid_twos:
#    i = centerlines[centerlines.SIFID == sifid_2].index.tolist()
#    if not i:
#        print(sifid_2)

display(nf1, nf2)

,FST_ROADNAME,SEC_ROADNAME,SIFCODE1,SIFCODE2,FST_SIFID,SEC_SIFID,GEOMETRY,state_grid_GEO
INTID,,,,,,,,
25314647488608,BRITTANY VALLEY RD,LIME KILN LN,0692,3860,689,3636,"[-85.63739842826769, 38.29902665283227]","[-85.63739357340009, 38.29901934737176]"
25533690815846,BRITTANY VALLEY RD,GLENVIEW AVE,0692,2566,689,2525,"[-85.6434753101308, 38.2964164239941]","[-85.64347045368233, 38.296409119316834]"
285078996590724,COFFEE TREE LN,COFFEE TREE PL,2053,2084,2053,2101,"[-85.53694581749606, 38.18196699123739]","[-85.53692786727781, 38.18195926142264]"
374478301589619,BOWLES AVE,WEBSTER ST,0624,7073,626,6361,"[-85.72680382327565, 38.25712141735226]","[-85.72679894542387, 38.25711412408772]"
374942158031000,BOWLES AVE,CABEL ST,0624,0898,626,875,"[-85.72832394001115, 38.256469323450425]","[-85.72831906176425, 38.256462030381684]"
414945534616712,CANDOR AVE,GARRS LN,0931,2488,903,2460,"[-85.81636719262018, 38.189799864851885]","[-85.81636229366431, 38.189792588666876]"
422723720394881,CANDOR AVE,LINHERK AVE,0931,3881,903,3654,"[-85.8169993940186, 38.18700350303334]","[-85.81699449508736, 38.18699622741413]"
432752417703459,BOWIE CT,BOWIE DR,0622,0623,624,625,"[-85.70260626299203, 38.14657969155578]","[-85.70260140045856, 38.1465724183819]"


,ROADNAME,SIFID,SIFCODE,low_cross_ROADNAME,SIFIDLOW,LOCROSSSIF,hi_cross_ROADNAME,SIFIDHI,HICROSSSIF,CORE_CLASS,GEOLOW,GEOHI
OBJECTID,,,,,,,,,,,,
3034,LIME KILN LN,3636,3860,LANSDOWNE AVE,3536,3743,RIVER KNOLLS DR,5040,5534,PRIMARY COLLECTOR,"[-85.64473369572579, 38.31284696929362]","[-85.64555838680533, 38.313566956021305]"
3125,WEBSTER ST,6361,7073,STORY AVE,13416,6183,BOWLES AVE,626,0624,LOCAL,"[-85.72647299204654, 38.25665296892628]","[-85.72680382327565, 38.25712141735226]"
3474,GARRS LN,2460,2488,CANDOR AVE,903,0931,NORTH LN,4265,4593,LOCAL,"[-85.81636719262018, 38.189799864851885]","[-85.81741529074715, 38.18987101606344]"
4249,GLENVIEW AVE,2525,2566,DUNRAVEN DR,1696,1726,DEAD END,8594,9811,LOCAL,"[-85.64201434345283, 38.29437496714594]","[-85.64219805634245, 38.29457377056122]"
4476,BOWIE DR,625,0623,BOWIE CT,624,0622,HICKOCK DR,13326,2913,LOCAL,"[-85.70260626299203, 38.14657969155578]","[-85.70169401763651, 38.14689512558871]"
...,...,...,...,...,...,...,...,...,...,...,...,...
31463,GLENVIEW AVE,2525,2566,GRAYSON CT,2626,2679,CABIN WAY,12896,1406,LOCAL,"[-85.64079071976235, 38.29284954144796]","[-85.64125447939766, 38.293433275538995]"
32265,COFFEE TREE PL,2101,2084,DEAD END,8594,9811,COFFEE TREE LN,2053,2053,LOCAL,"[-85.53795934713116, 38.181908231935815]","[-85.53694581749606, 38.18196699123739]"
32380,GARRS LN,2460,2488,LISA AVE,3661,3890,MILL CREEK DR,4085,4384,LOCAL,"[-85.81318601529698, 38.189609592766914]","[-85.81506403880756, 38.18972105387296]"


In [ ]:
# # # This is where Rehl Rd intersects Tucker station

""" SIFIDLOW	SIFIDHI	next	previouos
7109	5908	5908	NaN	NaN # next previous should be 12697?
9233	5908	12697	NaN	NaN
...
52804	12697	4674	NaN	NaN

This is where Rehl Rd intersects Tucker station

       | Tucker Station
       |
--Rehl--====Rehl====----Rehl--
                  |
                  | Tucker Station

This makes it too annoying to connect the segments using only SIFIDS. Lots of unusual cases.
We have exact geometry data, just use that
code for this already in centerline_data notebook """

' SIFIDLOW\tSIFIDHI\tnext\tpreviouos\n7109\t5908\t5908\tNaN\tNaN # next previous should be 12697?\n9233\t5908\t12697\tNaN\tNaN\n...\n52804\t12697\t4674\tNaN\tNaN\n\nThis is where Rehl Rd intersects Tucker station\n\n       | Tucker Station\n       |\n--Rehl--====Rehl====----Rehl--\n                  |\n                  | Tucker Station\n\nThis makes it too annoying to connect the segments using only SIFIDS. Lots of unusual cases.\nWe have exact geometry data, just use that\ncode for this already in centerline_data notebook '